# 06 — Approved Model Test Submission

This notebook scores `data/processed/test_features.csv` with the model approved at the end of notebook 05 and writes `data/output/submission.csv`. The true label is isolated before scoring and attached only after prediction.

## Inference contract

1. Load only an artifact marked `approved_for_inference`.
2. Load `test_features.csv` and preserve `default_flag` separately as `true_label`.
3. Remove the target and leakage columns before the model sees the data.
4. Quarantine or reject invalid rows according to `configs/inference.yaml`.
5. Apply the threshold and risk bands frozen inside the model artifact.
6. Export identifiers, probabilities, predicted classes, true labels and model traceability to `submission.csv`.

In [ ]:
import pandas as pd
from IPython.display import display

from bnpl_credit_risk.pipelines.batch_inference_pipeline import (
    BatchInferencePipeline,
)
from bnpl_credit_risk.settings import get_settings, load_config
from bnpl_credit_risk.visualization.inference import (
    BatchInferenceVisualizer,
)

settings = get_settings()
config = load_config()
batch_pipeline = BatchInferencePipeline(config=config, settings=settings)
batch_visualizer = BatchInferenceVisualizer(
    size=config.inference.visualization.size
)

## 1. Verify the deployed model

This cell intentionally fails if `latest` still points to a legacy or unapproved model. In that case, complete the publication cell at the end of notebook 05.

In [ ]:
model_contract = batch_pipeline.describe_model()
display(pd.Series(model_contract, name="Published model contract").to_frame())

if not model_contract["ready_for_scoring"]:
    print("\nSCORING BLOCKED")
    print(model_contract["blocking_reason"])

## 2. Load the labeled test partition

The complete test partition is used. Its target is retained only for the final comparison and is automatically removed by the inference layer before prediction.

In [ ]:
input_path = batch_pipeline.input_path
test_df = batch_pipeline.load_applications(input_path)

print(f"Input path: {input_path}")
print(f"Test shape: {test_df.shape}")
print(f"True label column: {config.data.target_column}")
display(test_df.head())

## 3. Build `submission.csv`

Validation, leakage protection, target isolation, scoring, label reconciliation and atomic output writing are handled by `BatchInferencePipeline`.

In [ ]:
submission_result = None
if not model_contract["ready_for_scoring"]:
    print("Submission skipped: publish the approved model from notebook 05.")
else:
    submission_result = batch_pipeline.run_test_submission()
    display(
        pd.Series(
            submission_result.summary, name="Submission summary"
        ).to_frame()
    )
    display(submission_result.submission.head())

## 4. Inspect predictions and true labels

Le CSV contient l’identifiant, la probabilité, `default_risk_class`, son libellé, le vrai label, `prediction_correct`, le seuil, la bande de risque, la version et l’horodatage. `predicted_label` reste un alias de compatibilité.

In [ ]:
predictions = None
if submission_result is None:
    print("Inspection skipped: no approved submission.")
else:
    predictions = submission_result.detailed_predictions
    display(submission_result.submission)
    display(
        predictions.sort_values("default_probability", ascending=False).head(10)
    )

## 5. Portfolio scoring overview

The figure combines risk bands, predicted default probabilities and the threshold-dependent decision distribution.

In [ ]:
if predictions is None:
    print("Portfolio overview skipped: no approved predictions.")
else:
    display(
        batch_visualizer.portfolio_overview(
            predictions,
            threshold=submission_result.decision_threshold,
            risk_band_order=[band.name for band in config.training.risk_bands],
        )
    )